In [2]:
import re
import string
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from collections import Counter

# 1. Load dataset
df = pd.read_csv("customer_complaints_1.csv")

# 2. Keep only non-null complaint text rows
df = df[['text']].dropna()

# 3. Rename column to Text for easier handling
df = df.rename(columns={'text': 'Text'})

# 4. Preprocessing function
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = text.split()
    tokens = [word for word in tokens if word not in ENGLISH_STOP_WORDS]
    return " ".join(tokens)

# 5. Apply preprocessing
df["Cleaned_Text"] = df["Text"].apply(preprocess_text)

# 6. Remove empty rows after preprocessing
df = df[df["Cleaned_Text"].str.strip() != ""]

# 7. TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(df["Cleaned_Text"])

# 8. KMeans clustering
k = 3
km = KMeans(n_clusters=k, random_state=42, n_init=10)
df["Cluster"] = km.fit_predict(X)

# 9. Show sample results
print(df[["Text", "Cleaned_Text", "Cluster"]].head(10))

# 10. Top terms per cluster
print("\nTop terms per cluster:")
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()

for i in range(k):
    print(f"\nCluster {i}:")
    for ind in order_centroids[i, :10]:
        print(terms[ind])

# 11. Purity calculation based on lab format
y_pred = df["Cluster"]
total_samples = len(y_pred)
cluster_label_counts = [Counter(y_pred)]
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("\nPurity:", purity)

                                                Text  \
0  I used to love Comcast. Until all these consta...   
1  I'm so over Comcast! The worst internet provid...   
2  If I could give them a negative star or no sta...   
3  I've had the worst experiences so far since in...   
4  Check your contract when you sign up for Comca...   
5  Thank God. I am changing to Dish. They gave me...   
6  I Have been a long time customer and only have...   
7  There is a malfunction on the DVR manager whic...   
8  Charges overwhelming. Comcast service rep was ...   
9  I have had cable, DISH, and U-verse, etc. in t...   

                                        Cleaned_Text  Cluster  
0  used love comcast constant updates internet ca...        1  
1  im comcast worst internet provider im taking o...        1  
2  negative star stars review worked industry bad...        2  
3  ive worst experiences far install problems sho...        1  
4  check contract sign comcast advertised offers ...        1  